In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Iterative Computation of Powers via Iterative Squaring

As the powers of numbers get big fast, we use the module `Z` of the library Zarith, which implements integers of arbitrary size.  Their type is `Z.t`.

In [ ]:
#require "zarith.top";;

The function `power` takes two arguments:
- `x0` is an integer of type `Z.t`, the *base*,
- `y0` is a natural number of type `int`, the *exponent*.

The function returns the power $x_0^{y_0}$.

**Implementation:** As the parameters of a function can not be changed in OCaml, we use the *references* `x`, `y`, and `r`.  The references `x` and `y` are initialized with the values of the parameters `x0` and `y0`.  The function maintains the *invariant*
$$ r_n \cdot x_n^{y_n} = x_0^{y_0}. $$
Here $x_n$, $y_n$, and $r_n$ are the values of the references `x`, `y`, and `r` after $n$ iterations of the loop.  Note that we have:
* $x_{n+1} = x_n^2$,
* $y_{n+1} = y_n \;\texttt{/}\; 2$,
* $r_{n+1} = \left\{\begin{array}{ll}
                     r_n \cdot x_n & \texttt{if} \; y_n \;\texttt{mod}\; 2 = 1, \\
                     r_n           & \texttt{if} \; y_n \;\texttt{mod}\; 2 = 0. \\
                     \end{array}\right.
  $

Here, $y_n \;\texttt{/}\; 2$ denotes the integer division of $y_n$ by $2$.
When the loop terminates, we have $y_n = 0$ and hence the invariant implies $r_n = x_0^{y_0}$.

Inside the *local open* `Z.(...)`, the operator `*` denotes the multiplication of numbers of type `Z.t`, while `!` still denotes the content of a reference.  `Z.one` is the number $1$ of type `Z.t`.

In [ ]:
let power (x0 : Z.t) (y0 : int) : Z.t =
  let x = ref x0 in
  let y = ref y0 in
  let r = ref Z.one in
  while !y > 0 do
    if !y mod 2 = 1 then
      r := Z.(!r * !x);
    x := Z.(!x * !x);
    y := !y / 2
  done;
  !r

We compute $3^{500\,000}$.  This number has more than $200\,000$ digits.  Therefore, we only display the number of its digits.  The result `p` is defined locally with `let ... in`, since the toplevel would display all digits of a variable defined with `let` at the top level.

The function `time`, which is defined in the file `setup.ml`, takes two arguments:
- `label` is a string,
- `f` is a function whose only argument is `()`.

It computes `f ()`, prints `label` together with the time this computation took, and returns the result of `f ()`.  The computation to be measured is wrapped into an anonymous function `fun () -> ...`, so that it is only executed when `time` calls `f`.

In [ ]:
let p = time "power 3 500_000" (fun () -> power (Z.of_int 3) 500_000) in
String.length (Z.to_string p)

In [ ]:
power (Z.of_int 3) 5

We test the function `power` using random numbers.  We compare it with the function `Z.pow` from the library Zarith.  The function call `Random.int b` returns a random number $x$ such that $0 \leq x < b$.  When the kernel starts, the random number generator of OCaml is initialized with a fixed seed.  Hence, the tests are reproducible if the notebook is executed from the beginning.  The function `failwith` raises the exception `Failure` with the given message, and `Z.equal` checks whether two numbers of type `Z.t` are equal.

In [ ]:
time "random checks" (fun () ->
  for _ = 1 to 100_000 do
    let a = Z.of_int (1 + Random.int 1_000) in
    let b = Random.int 1_000 in
    if not (Z.equal (power a b) (Z.pow a b)) then
      failwith (Printf.sprintf "Mismatch for a=%s, b=%d" (Z.to_string a) b)
  done);
print_endline "All tests have been successful!"